### Generates scatter plots to visualize solver failure points.

This script queries a benchmark database to identify interior-point method (IPM) iterations where the solver's residual norm exceeded the PCG tolerance. It then creates a grid of scatter plots, one for each solver, showing these failure points as a function of the problem size (number of edges).

In [23]:
import sqlite3
import polars as pl
import matplotlib.pyplot as plt
import argparse
import re
import numpy as np

In [24]:
def plot_failure_points(db_path, problem_pattern=None, solver_pattern=None):
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    query = """
    WITH data AS (
        SELECT
            p.name AS problem_name,
            p.num_edges,
            r.solver_name,
            sh.ipm_iter,
            sh.absolute_residual_norm,
            sh.relative_residual_norm,
            COALESCE(c.pcg_tol, 1e-6) AS pcg_tol
        FROM
            solver_history sh
        JOIN
            runs r ON sh.run_id = r.id
        JOIN
            problems p ON r.problem_id = p.id
        JOIN
            configs c ON r.config_id = c.id
    )
    SELECT *
    FROM data
    WHERE relative_residual_norm > pcg_tol
    ORDER BY problem_name, solver_name, ipm_iter;
    """
    df = pl.read_database(query, conn)
    conn.close()
    print(f"Data loaded from database: {df}")

    if df.is_empty():
        print("No failures found matching the criteria.")
        return

    if problem_pattern:
        df = df.filter(
            pl.col("problem_name").str.contains(problem_pattern, strict=False)
        )
    if solver_pattern:
        df = df.filter(pl.col("solver_name").str.contains(solver_pattern, strict=False))

    if df.is_empty():
        print(
            f"No data found after applying filters (problem_pattern='{problem_pattern}', solver_pattern='{solver_pattern}')."
        )
        return

    unique_solvers = df["solver_name"]
    unique_problems = df["problem_name"]

    colors = plt.get_cmap("tab20", len(unique_problems))
    problem_color_map = {
        problem: colors(i) for i, problem in enumerate(unique_problems)
    }

    num_solvers = len(unique_solvers)
    if num_solvers == 0:
        print("No solvers with failures to plot.")
        return

    cols = min(num_solvers, 2)
    rows = (num_solvers + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
    axes = axes.flatten()

    for i, solver in enumerate(unique_solvers):
        ax = axes[i]
        solver_df = df.filter(pl.col("solver_name") == solver)

        for problem in unique_problems:
            problem_solver_df = solver_df.filter(pl.col("problem_name") == problem)
            if not problem_solver_df.is_empty():
                ax.scatter(
                    problem_solver_df["num_edges"]
                    , problem_solver_df["ipm_iter"]
                    , color=problem_color_map[problem]
                    , alpha=0.7
                    , s=20
                )

        ax.set_xscale("log")
        ax.set_xlabel("Number of Edges (log scale)", fontsize=9)
        ax.set_ylabel("IPM Iteration", fontsize=9)
        ax.set_title(f"Solver: {solver}", fontsize=10)
        ax.grid(True, which="both", ls="--")
        ax.tick_params(axis="both", which="major", labelsize=8)
        ax.tick_params(axis="both", which="minor", labelsize=6)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle("IPM Iterations at which Residual Norm > pcg_tol", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [25]:
plot_failure_points("../warmup.db") # Assuming a default db file for demonstration


Data loaded from database: shape: (0, 7)
┌──────────────┬───────────┬─────────────┬──────────┬──────────────────┬─────────────────┬─────────┐
│ problem_name ┆ num_edges ┆ solver_name ┆ ipm_iter ┆ absolute_residua ┆ relative_residu ┆ pcg_tol │
│ ---          ┆ ---       ┆ ---         ┆ ---      ┆ l_norm           ┆ al_norm         ┆ ---     │
│ null         ┆ null      ┆ null        ┆ null     ┆ ---              ┆ ---             ┆ null    │
│              ┆           ┆             ┆          ┆ null             ┆ null            ┆         │
╞══════════════╪═══════════╪═════════════╪══════════╪══════════════════╪═════════════════╪═════════╡
└──────────────┴───────────┴─────────────┴──────────┴──────────────────┴─────────────────┴─────────┘
No failures found matching the criteria.
